# Find MPAS boundary cells and simplify the domain polygon
- Find all boundary cells from `bdyMaskCell` in the MPAS grid file
- Plot boundary cell shapes
- Use `shapely.geometry.Polygon.simplify` to reduce thousands of boundary points to a few dozen
- Verify the simplified polygon covers the whole domain

## get the `pyDAmonitor_ROOT` env variable
This step is highly recommended. It is required if one want to use the DAmonitor Python package or use the MPAS/FV3 sample data or local cartopy nature_earth_data.

In [ ]:
%%time
# autoload external python modules if they changed
%load_ext autoreload
%autoreload 2
    
import sys, os
pyDAmonitor_ROOT=os.getenv("pyDAmonitor_ROOT")
if pyDAmonitor_ROOT is None:
    print("!!! pyDAmonitor_ROOT is NOT set. Run `source ush/load_pyDAmonitor.sh`")
else:
    print(f"pyDAmonitor_ROOT={pyDAmonitor_ROOT}\n")
sys.path.insert(0, pyDAmonitor_ROOT)

## import modules

In [ ]:
%%time
import numpy as np
from netCDF4 import Dataset

import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Polygon, MultiPoint
from shapely import concave_hull
from DAmonitor.base import query_dataset
cartopy.config['data_dir'] = f"{pyDAmonitor_ROOT}/data/natural_earth_data"

## read in the MPAS grid file and inspect `bdyMaskCell`, extract outermost boundary cells and their vertices

In [ ]:
%%time
mpas_domain = "conus3km"
grid_file = os.path.join(pyDAmonitor_ROOT, f'data/mpasjedi/{mpas_domain}.grid.nc')
ds = Dataset(grid_file, 'r')

bdyMaskCell = ds.variables['bdyMaskCell'][:]
print(f"bdyMaskCell unique values: {np.unique(bdyMaskCell)}")
print(f"  0 (interior): {np.sum(bdyMaskCell == 0)} cells")
for v in range(1, bdyMaskCell.max() + 1):
    print(f"  {v} (boundary layer {v}): {np.sum(bdyMaskCell == v)} cells")

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# outermost boundary: maximum value of bdyMaskCell
bdy_max = bdyMaskCell.max()

# cell center coordinates (convert radians to degrees, lon to -180..180)
lonCell = np.degrees(ds.variables['lonCell'][:])
latCell = np.degrees(ds.variables['latCell'][:])
lonCell = np.where(lonCell > 180, lonCell - 360, lonCell)

# vertex coordinates
lonVertex = np.degrees(ds.variables['lonVertex'][:])
latVertex = np.degrees(ds.variables['latVertex'][:])
lonVertex = np.where(lonVertex > 180, lonVertex - 360, lonVertex)

# connectivity
verticesOnCell = ds.variables['verticesOnCell'][:]  # (nCells, maxEdges), 1-based
nEdgesOnCell = ds.variables['nEdgesOnCell'][:]

# all boundary cells (bdyMaskCell >= 1) and per-layer masks
mask_all_bdy = bdyMaskCell >= 1
bdy_layers = {}
for layer in range(1, bdy_max + 1):
    bdy_layers[layer] = bdyMaskCell == layer

# outermost boundary cell data (for polygon construction)
mask_outer = bdyMaskCell == bdy_max
lat_bdy = latCell[mask_outer]
lon_bdy = lonCell[mask_outer]
verts_bdy = verticesOnCell[mask_outer]
nEdges_bdy = nEdgesOnCell[mask_outer]

print(f"Total boundary cells (all {bdy_max} layers): {mask_all_bdy.sum()}")
print(f"Outermost boundary cells (layer {bdy_max}): {mask_outer.sum()}")
print(f"Lat range: {lat_bdy.min():.2f} to {lat_bdy.max():.2f}")
print(f"Lon range: {lon_bdy.min():.2f} to {lon_bdy.max():.2f}")

## build a polygon from boundary cell **vertices** and simplify
Use outer vertices (not cell centers) so the polygon covers entire cells, not just their centers. Then `simplify` to reduce to a few dozen vertices.

In [ ]:
%%time
# collect all unique vertices of outermost boundary cells
# these vertices are at the cell edges, extending beyond cell centers
vertex_ids = set()
for i in range(mask_outer.sum()):
    nEdges = nEdges_bdy[i]
    for j in range(nEdges):
        vertex_ids.add(verts_bdy[i, j] - 1)  # convert 1-based to 0-based

vertex_ids = np.array(sorted(vertex_ids))
bdy_vlons = lonVertex[vertex_ids]
bdy_vlats = latVertex[vertex_ids]
print(f"Unique vertices on outermost boundary cells: {len(vertex_ids)}")

# create a concave hull from boundary cell vertices
points = MultiPoint(list(zip(bdy_vlons, bdy_vlats)))
concave_ratio = 0.3  # 0.3 gives a tighter fit than convex hull; adjust as needed (the smaller the better fit to convave shapes)
domain_polygon = concave_hull(points, ratio=0.3)

print(f"Domain polygon type: {domain_polygon.geom_type}")
print(f"Number of boundary points (before simplify): {len(domain_polygon.exterior.coords)}")

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# # buffer_deg: buffer the polygon outward so that after simplification all cells remain inside
buffer_deg = 0.2  # degrees; increase if cells still fall outside after simplify
domain_polygon_buffered = domain_polygon.buffer(buffer_deg)
print(f"Buffered polygon points: {len(domain_polygon_buffered.exterior.coords)}")

# simplify the buffered polygon
tolerance = 0.15  # degrees; the larger the less points
simplified = domain_polygon_buffered.simplify(tolerance, preserve_topology=True)

print(f"Number of points after simplify (tol={tolerance}): {len(simplified.exterior.coords)}")

# try different tolerances to find the sweet spot
for tol in [0.1, 0.15, 0.3, 0.5, 1.0, 1.5, 2.0]:
    s = domain_polygon_buffered.simplify(tol, preserve_topology=True)
    print(f"  tolerance={tol:.2f} => {len(s.exterior.coords)} points")

## plot original and simplified polygon as well as outermost cells (interactive)

In [ ]:
fig = go.Figure()

# outermost boundary cell polygons (single trace with None separators)
all_lats = []
all_lons = []
for i in range(lat_bdy.size):
    nEdges = nEdges_bdy[i]
    verts = verts_bdy[i, :nEdges] - 1
    poly_lon = list(lonVertex[verts]) + [lonVertex[verts[0]]]
    poly_lat = list(latVertex[verts]) + [latVertex[verts[0]]]
    all_lats.extend(poly_lat + [None])
    all_lons.extend(poly_lon + [None])

fig.add_trace(go.Scattermap(
    lat=all_lats, lon=all_lons,
    mode='lines',
    line=dict(width=1, color='gray'),
    name='Boundary cells',
    hoverinfo='skip',
))

# original concave hull
ox, oy = domain_polygon.exterior.xy
fig.add_trace(go.Scattermap(
    lat=list(oy), lon=list(ox),
    mode='lines',
    line=dict(width=2, color='blue'),
    name='Original boundary',
))

# simplified polygon
sx, sy = simplified.exterior.xy
fig.add_trace(go.Scattermap(
    lat=list(sy), lon=list(sx),
    mode='lines+markers',
    line=dict(width=3, color='red'),
    marker=dict(size=7, color='red'),
    name=f'Simplified ({len(simplified.exterior.coords)} pts)',
))

fig.update_layout(
    width=1600,
    height=1200,
    margin=dict(l=0, r=0, t=40, b=0),
    title="Domain boundary: original vs simplified",
    map_style="open-street-map",
    map_center=dict(lat=float(np.mean(lat_bdy)), lon=float(np.mean(lon_bdy))),
    map_zoom=3,
)
fig.show()

## print the simplified polygon vertices for reference

In [ ]:
coords = list(simplified_buffered.exterior.coords)
print(f"Simplified polygon has {len(coords)} vertices (including closing point):")
print(f"{'idx':>4s}  {'lon':>9s}  {'lat':>9s}")
for i, (x, y) in enumerate(coords):
    print(f"{i:4d}  {x:9.4f}  {y:9.4f}")